In [5]:
import pandas as pd
import numpy as np

# Carregar a base de dados
df = pd.read_csv("../data/Telco-Customer-Churn.csv")

print(f"Dimensões da base: {df.shape[0]} registros e {df.shape[1]} atributos\n")

# Verificar tipos de dados e nulos aparentes
print("--- INFORMAÇÕES GERAIS E TIPOS ---")
print(df.info())

# Investigar duplicatas e nulos ocultos (strings vazias)
print(f"\nLinhas duplicadas: {df.duplicated().sum()}")
espacos_vazios = (df == " ").sum()
print("\n--- VALORES COM ESPAÇO EM BRANCO (' ') ---")
print(espacos_vazios[espacos_vazios > 0])

# Distribuição da Variável-Alvo (Churn)
print("\n--- DISTRIBUIÇÃO DA VARIÁVEL-ALVO (CHURN) ---")
print(df["Churn"].value_counts())
print("\nProporção (%):")
print((df["Churn"].value_counts(normalize=True) * 100).round(2))

# Estatísticas das variáveis numéricas
print("\n--- ESTATÍSTICAS NUMÉRICAS ---")
print(df.describe())

Dimensões da base: 7043 registros e 21 atributos

--- INFORMAÇÕES GERAIS E TIPOS ---
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract       

In [7]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import chi2, f_classif
from sklearn.preprocessing import OrdinalEncoder

# 1. Prepara a cópia limpa para os testes estatísticos
df_eda = df.copy()
df_eda["TotalCharges"] = pd.to_numeric(df_eda["TotalCharges"], errors="coerce")
df_eda = df_eda.drop(columns=["customerID"])

# Remove temporariamente as 11 linhas com NaN em TotalCharges apenas para calcular os testes
df_temp = df_eda.dropna().copy()
y_num = df_temp["Churn"].map({"No": 0, "Yes": 1})

# 2. Define as colunas numéricas e categóricas
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in df_temp.columns if c not in num_cols + ["Churn"]]

# 3. Executa ANOVA F-test (numéricas) e Qui-Quadrado (categóricas)
f_scores, p_values_num = f_classif(df_temp[num_cols], y_num)
X_cat_enc = OrdinalEncoder().fit_transform(df_temp[cat_cols])
chi_scores, p_values_cat = chi2(X_cat_enc, y_num)

# 4. Monta a tabela unificada de decisão (critério: alpha = 0.05)
res_num = pd.DataFrame({
    "Atributo": num_cols,
    "Tipo": "Numérica",
    "Teste": "ANOVA F-test",
    "Estatística": f_scores,
    "p_value": p_values_num
})

res_cat = pd.DataFrame({
    "Atributo": cat_cols,
    "Tipo": "Categórica",
    "Teste": "Qui-Quadrado (χ²)",
    "Estatística": chi_scores,
    "p_value": p_values_cat
})

df_selecao = pd.concat([res_num, res_cat], ignore_index=True)
df_selecao["Decisão"] = np.where(df_selecao["p_value"] < 0.05, "Mantida", "Descartada")
df_selecao = df_selecao.sort_values(by="p_value").reset_index(drop=True)

pd.set_option("display.max_rows", 30)
display(df_selecao)

,Atributo,Tipo,Teste,Estatística,p_value,Decisão
0,Contract,Categórica,Qui-Quadrado (χ²),1111.759054,9.186154e-244,Mantida
1,tenure,Numérica,ANOVA F-test,1007.509431,9.437650e-207,Mantida
2,OnlineSecurity,Categórica,Qui-Quadrado (χ²),550.437364,1.011220e-121,Mantida
3,TechSupport,Categórica,Qui-Quadrado (χ²),522.164716,1.430807e-115,Mantida
4,TotalCharges,Numérica,ANOVA F-test,291.344862,4.876866e-64,Mantida
5,MonthlyCharges,Numérica,ANOVA F-test,271.576990,6.760843e-60,Mantida
6,OnlineBackup,Categórica,Qui-Quadrado (χ²),229.441123,7.892689e-52,Mantida
7,DeviceProtection,Categórica,Qui-Quadrado (χ²),190.688004,2.249863e-43,Mantida
8,SeniorCitizen,Categórica,Qui-Quadrado (χ²),133.482766,7.089546e-31,Mantida
9,Dependents,Categórica,Qui-Quadrado (χ²),131.271509,2.159540e-30,Mantida


In [8]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração de estilo e diretório de saída
sns.set_theme(style="whitegrid")
ASSETS_DIR = Path("../assets")
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

# 1. Salva a tabela de seleção de variáveis para usar no Streamlit e no Relatório
df_selecao.to_csv(ASSETS_DIR / "tabela_selecao_variaveis.csv", index=False)

# 2. Diagnóstico objetivo de Outliers (Critério IQR) nas variáveis numéricas
print("=== DIAGNÓSTICO DE OUTLIERS (CRITÉRIO IQR) ===")
for col in num_cols:
    Q1 = df_temp[col].quantile(0.25)
    Q3 = df_temp[col].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    outliers = df_temp[(df_temp[col] < lim_inf) | (df_temp[col] > lim_sup)]
    print(f"- {col}: {len(outliers)} outliers (Limites: [{lim_inf:.2f}, {lim_sup:.2f}] | Min/Max reais: [{df_temp[col].min():.2f}, {df_temp[col].max():.2f}])")

# 3. Separa as 17 variáveis mantidas por tipo
selected_num = df_selecao[(df_selecao["Tipo"] == "Numérica") & (df_selecao["Decisão"] == "Mantida")]["Atributo"].tolist()
selected_cat = df_selecao[(df_selecao["Tipo"] == "Categórica") & (df_selecao["Decisão"] == "Mantida")]["Atributo"].tolist()

# 4. Gera e salva os 3 gráficos das variáveis numéricas vs. Churn (Boxplot)
for col in selected_num:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    sns.boxplot(
        data=df_temp, x="Churn", y=col, hue="Churn",
        palette={"No": "#2b5c8f", "Yes": "#d95f02"}, legend=False, ax=ax
    )
    ax.set_title(f"Distribuição de {col} por Status de Churn", fontsize=12, fontweight="bold")
    ax.set_xlabel("Churn (Cancelamento)")
    ax.set_ylabel(col)
    plt.tight_layout()
    plt.savefig(ASSETS_DIR / f"eda_num_{col}.png", dpi=150)
    plt.close()

# 5. Gera e salva os 14 gráficos das variáveis categóricas vs. Churn (Barras Empilhadas %)
for col in selected_cat:
    ct = pd.crosstab(df_temp[col], df_temp["Churn"], normalize="index") * 100
    ax = ct[["No", "Yes"]].plot(
        kind="bar", stacked=True, figsize=(7.5, 4.5),
        color=["#2b5c8f", "#d95f02"], edgecolor="white"
    )
    ax.set_title(f"Taxa de Churn (%) por {col}", fontsize=12, fontweight="bold")
    ax.set_xlabel(col)
    ax.set_ylabel("Percentual de Clientes (%)")
    ax.legend(title="Churn", labels=["Não (No)", "Sim (Yes)"], loc="upper right")
    plt.xticks(rotation=25, ha="right")
    
    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f%%", label_type="center", color="white", fontsize=9, fontweight="bold")
        
    plt.tight_layout()
    plt.savefig(ASSETS_DIR / f"eda_cat_{col}.png", dpi=150)
    plt.close()

print(f"\nSucesso! {len(selected_num) + len(selected_cat)} gráficos e 1 CSV salvos em: {ASSETS_DIR.resolve()}")

=== DIAGNÓSTICO DE OUTLIERS (CRITÉRIO IQR) ===
- tenure: 0 outliers (Limites: [-60.00, 124.00] | Min/Max reais: [1.00, 72.00])
- MonthlyCharges: 0 outliers (Limites: [-45.82, 171.27] | Min/Max reais: [18.25, 118.75])
- TotalCharges: 0 outliers (Limites: [-4688.48, 8884.67] | Min/Max reais: [18.80, 8684.80])

Sucesso! 17 gráficos e 1 CSV salvos em: /home/tharles-mazurkievicz/Documentos/ProjetoDados/assets
